# Ingestion Exploration — Phase 2

**Purpose:** Inspect raw text extracted by `app/ingestion/parser.py` on the real Fall 2026 Prospectus PDF, *before* writing the cleaning logic.

We are specifically looking for:
- Repeated headers / footers (e.g. document title, page numbers) that appear on every page
- Broken line breaks (words split across lines)
- Extra whitespace / weird characters

This notebook does **not** modify any pipeline code — it's a read-only inspection step. Findings here will directly inform what `app/ingestion/cleaner.py` needs to handle.

In [ ]:
import sys
from pathlib import Path

# Make `app` importable when running this notebook from the notebooks/ folder
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.ingestion.parser import extract_pdf_pages, summarize_extraction

PDF_PATH = project_root / "data" / "raw_documents" / "Prospectus - FALL 2026 (29-07-2026).pdf"

pages = extract_pdf_pages(PDF_PATH)
summary = summarize_extraction(pages)

print(f"Loaded: {PDF_PATH.name}")
print(f"Total pages: {summary['total_pages']}")
print(f"Empty pages: {summary['empty_pages']}")
print(f"Low-text pages: {summary['low_text_pages']}")
print(f"Avg chars/page: {summary['avg_chars_per_page']}")

## 1. Raw text sample — first, middle, and last page

Sampling across the document (not just page 1) because prospectuses often have a very different layout on cover pages vs. body pages vs. index/appendix pages.

In [ ]:
def show_page(page: dict, max_chars: int = 1200) -> None:
    text = page["text"]
    truncated = text[:max_chars]
    suffix = "\n... [truncated]" if len(text) > max_chars else ""

    print(f"{'=' * 70}")
    print(f"PAGE {page['page_number']}  |  char_count = {page['char_count']}")
    print(f"{'=' * 70}")
    print(truncated + suffix)
    print()


sample_indices = [0, len(pages) // 2, len(pages) - 1]  # first, middle, last

for idx in sample_indices:
    show_page(pages[idx])

## 2. Header / footer pattern detection

For a set of pages, print just the **first 2 lines** and **last 2 lines** of each. If the same (or near-identical) text repeats across many pages, that is almost certainly a running header or footer that `cleaner.py` should strip out.

In [ ]:
import random

random.seed(42)  # reproducible sample
sample_size = 10
sampled_pages = random.sample(pages, k=min(sample_size, len(pages)))
sampled_pages.sort(key=lambda p: p["page_number"])

print(f"{'PAGE':<6} | {'FIRST 2 LINES':<50} | {'LAST 2 LINES'}")
print("-" * 110)

for page in sampled_pages:
    lines = [l.strip() for l in page["text"].splitlines() if l.strip()]
    first_two = " | ".join(lines[:2]) if lines else "(empty)"
    last_two = " | ".join(lines[-2:]) if lines else "(empty)"
    print(f"{page['page_number']:<6} | {first_two[:50]:<50} | {last_two[:50]}")

## 3. Findings (fill in after reviewing output above)

- [x] Repeated header text: "UNIVERSITY OF EDUCATION, LAHORE" + "FALL ..." on ~50% of sampled pages
- [x] Repeated footer text: none found
- [x] Broken line breaks observed? No
- [x] Any weird characters / encoding issues? No

These findings defined the exact rules implemented in `app/ingestion/cleaner.py`.

## 4. Cleaner verification — before vs. after

`app/ingestion/cleaner.py` has been implemented based on the findings above. This section visually confirms it works correctly on:
- **Page 13** — a page that HAD the running header (should be removed)
- **Page 53** — a page that did NOT have the header (should be left basically untouched, only whitespace normalized)

In [ ]:
from app.ingestion.cleaner import clean_pages

cleaned_pages = clean_pages(pages)


def show_before_after(page_number: int, max_chars: int = 500) -> None:
    page = cleaned_pages[page_number - 1]
    print(f"{'#' * 70}")
    print(f"PAGE {page_number}")
    print(f"{'#' * 70}")
    print(f"--- BEFORE (raw, {page['char_count']} chars) ---")
    print(page["text"][:max_chars])
    print()
    print(f"--- AFTER (cleaned, {page['cleaned_char_count']} chars) ---")
    print(page["cleaned_text"][:max_chars])
    print()


show_before_after(13)   # had the header -> should be gone
show_before_after(53)   # no header -> should look almost identical

In [ ]:
# Document-wide sanity check: how many pages still contain the header
# text after cleaning? Should be 0.
still_has_header = [
    p["page_number"] for p in cleaned_pages
    if "UNIVERSITY OF EDUCATION, LAHORE" in p["cleaned_text"]
]

total_raw_chars = sum(p["char_count"] for p in cleaned_pages)
total_cleaned_chars = sum(p["cleaned_char_count"] for p in cleaned_pages)
removed_chars = total_raw_chars - total_cleaned_chars

print(f"Pages still containing header after cleaning: {len(still_has_header)}")
print(f"Total raw chars: {total_raw_chars}")
print(f"Total cleaned chars: {total_cleaned_chars}")
print(f"Chars removed: {removed_chars} ({removed_chars / total_raw_chars:.1%} of document)")